# soulclip on Colab — a 5-minute AI film in under an hour

Turns a scene script into a stitched film using **Wan 2.1 T2V 1.3B** plus the
**CausVid step-distilled LoRA** on Colab's free T4. No API key, no payment.

**Before you start:** `Runtime > Change runtime type > T4 GPU`.

### Why `--fast` changes everything

The default settings run 20 denoising steps *with* classifier-free guidance,
and CFG runs the model twice per step — 40 forward passes per clip.

CausVid is distilled to work in **4 steps at guidance 1.0**, so CFG disappears
entirely: **4 passes instead of 40**.

| Setting | Passes/clip | 60 clips |
|---|---|---|
| 20 steps + CFG (default) | 40 | ~8 hours |
| 10 steps + CFG | 20 | ~4 hours |
| **`--fast` (CausVid), 832x480** | **4** | **~50 min** |
| **`--fast`, 640x368** | **4** | **~30 min** |

### Colab free-tier limits

| | |
|---|---|
| Max session | 12 hours while actively computing |
| Idle timeout | ~90 min (only when nothing is running) |
| Weekly GPU quota | ~15-30 hours |

With `--fast` the whole film fits comfortably inside one session.

**Quality note:** 4-step output is softer, with less fine detail and slightly
simpler motion than 20-step. For anime-styled work it holds up well. Try 6
clips both ways and judge for yourself.


## 1. Check the GPU


In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), (
    'No GPU. Runtime > Change runtime type > T4 GPU, then rerun.')
p = torch.cuda.get_device_properties(0)
print(f'{p.name}, {p.total_memory/1e9:.1f} GB VRAM')


## 2. Install

Takes 2-3 minutes.


In [ ]:
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg
!git clone -q https://github.com/Naserkhan07/soul_exter.git 2>/dev/null || true
%cd /content/soul_exter
!git checkout -q arena/019f98a2-soul-exter && git pull -q
print('ready')


## 3. Keep your work across disconnects (recommended)

Saves clips to Drive so a dropped session costs you nothing. Skip this
cell if you would rather not connect Drive — but then a disconnect loses
everything generated so far.


In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKDIR = '/content/drive/MyDrive/soulclip/work'
    OUTPUT  = '/content/drive/MyDrive/soulclip/film.mp4'
else:
    WORKDIR = '/content/work'
    OUTPUT  = '/content/film.mp4'

print('clips ->', WORKDIR)
print('film  ->', OUTPUT)


## 4. Your script

Label scenes `Scene 1:`, `Scene 2:` ... or separate them with blank lines.

**Prompt tips for Wan:** describe the *camera* and the *motion*, not just
the subject — 'slow dolly in', 'waves crash', 'hair moves in the wind'.
Repeat character details in every scene; the model has no memory between
clips.


In [ ]:
script = '''
Scene 1: A lighthouse on a black rock headland at dusk, its beam sweeping
slowly across heavy grey water. Rain streaks sideways. Slow dolly in.

Scene 2: Inside the lantern room, brass fittings glowing warm. An old
keeper in a wool coat winds a mechanism by hand. Firelight flickers.

Scene 3: Waves crash white over a dark reef, spray flung high into the
storm. Handheld camera, violent motion.

Scene 4: A small fishing boat pinned against the rocks, mast broken, a
single lantern swinging wildly on the deck.

Scene 5: The keeper hauls a heavy lever with both hands, straining. The
great beam swings and holds steady.

Scene 6: Dawn over a calm flat sea, pale gold light. Two figures wrapped
in blankets sit on stone steps, steam rising from tin mugs.
'''

with open('/content/script.txt', 'w') as f:
    f.write(script)

!python -m soulclip.cli scenes /content/script.txt --clip-seconds 5


## 5. Settings

`CLIPS` is the main dial. Each clip is ~5 s, so 6 clips = 30 s of film.

Leave `STEPS` at 20 for good quality, or drop to 10 to roughly halve the
time at some cost in sharpness.


In [ ]:
CLIPS = 60           # 60 clips x 5.06s = ~5m05s film
FAST  = True         # CausVid LoRA: 4 steps, no CFG (~10x faster)
WIDTH, HEIGHT = 832, 480   # try 640x368 to roughly halve the time again

STYLE = 'cinematic anime, detailed background art, dramatic lighting, film grain'

secs_per_clip = 48 if FAST else 240
secs_per_clip *= (WIDTH*HEIGHT)/(832*480)
total = CLIPS*secs_per_clip/60
mins = CLIPS*5.06
print(f'{CLIPS} clips -> {int(mins//60)}m{int(mins%60):02d}s of film')
print(f'~{secs_per_clip:.0f}s per clip -> ~{total:.0f} min of generation')
print('plus ~8 min setup and ~1 min stitching')


## 6. Generate

The first run downloads ~6 GB of weights (a few minutes, once per session).

**If it disconnects, just run this cell again** — finished clips are reused
and only the missing ones are generated.


In [ ]:
FAST_FLAG = '--fast' if FAST else ''

!python -m soulclip.cli render /content/script.txt \
    --provider wan $FAST_FLAG \
    --clip-seconds 5 \
    --max-scenes $CLIPS \
    --target $((CLIPS*5)) \
    --width $WIDTH --height $HEIGHT \
    --style "$STYLE" \
    --workdir "$WORKDIR" \
    -o "$OUTPUT"


## 7. Watch it


In [ ]:
from IPython.display import HTML
from base64 import b64encode

data = b64encode(open(OUTPUT, 'rb').read()).decode()
HTML(f'<video width=640 controls src="data:video/mp4;base64,{data}"></video>')


## 8. Download


In [ ]:
from google.colab import files
files.download(OUTPUT)


---
## Timing reference

One Wan clip is **81 frames @ 16fps = 5.06 s**.

| | Clips | Final length |
|---|---|---|
| Hard cuts | **60** | 5m05s |
| 0.4 s crossfades | **65** | 5m03s |

Crossfades overlap clips (64 junctions x 0.4 s ≈ 26 s lost), so they need 5
extra clips for the same runtime. Both measured by stitching real files.

| Stage | `--fast` @832x480 | `--fast` @640x368 |
|---|---|---|
| Setup + download | ~8 min | ~8 min |
| Generation (60 clips) | ~48 min | ~28 min |
| Stitching (measured) | ~1 min | ~1 min |
| **TOTAL** | **~57 min** | **~37 min** |

### If it is still too slow

- `--wan-frames 49` → 3 s clips (need ~100 for 5 min, but each is ~40% cheaper)
- Run two notebooks on different scene ranges and merge — Colab allows 2
  concurrent sessions

### If you hit out-of-memory

Add `--wan-frames 33` or drop to `640x368`.

### Honest expectations

Wan 1.3B at 4 steps is the fastest usable configuration, not the best-looking
one. Output is 480p, softer than 20-step, and well short of paid Kling or Veo.
Characters will not stay consistent between shots — that is the model, not the
pipeline. Repeating character details in every scene helps a little.
